how many listings are in each neighbourhood?

what is the typical number of guests for each type of room?

In [43]:
import pandas as pd
listings = pd.read_csv('data/listings.csv')
listings.shape

(4574, 84)

count per group: value_counts

In [44]:
listings['room_type'].value_counts()

room_type
Entire home/apt    3760
Private room        787
Shared room          24
Hotel room            3
Name: count, dtype: int64

In [45]:
listings['room_type'].normalize(True)

AttributeError: 'Series' object has no attribute 'normalize'

In [ ]:
listings['neighbourhood_cleansed'].value_counts()

neighbourhood_cleansed
Södermalms                1145
Norrmalms                  469
Hägersten-Liljeholmens     464
Enskede-Årsta-Vantörs      445
Kungsholmens               420
Östermalms                 297
Bromma                     281
Skarpnäcks                 221
Älvsjö                     214
Farsta                     177
Hässelby-Vällingby         148
Rinkeby-Tensta             108
Skärholmens                 96
Spånga-Tensta               89
Name: count, dtype: int64

In [ ]:
listings['bedrooms'].value_counts(dropna=False).sort_index()

bedrooms
0.0      257
1.0     2490
2.0      739
3.0      588
4.0      351
5.0      116
6.0       23
7.0        3
8.0        1
12.0       4
NaN        2
Name: count, dtype: int64

In [ ]:
listings['bedrooms'].describe()

count    4572.000000
mean        1.734689
std         1.247283
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        12.000000
Name: bedrooms, dtype: float64

Group and calculate: groupby
1. split
2. calculate
3. combine

In [ ]:
listings.groupby('room_type')['accommodates'].mean().round(2)

room_type
Entire home/apt    3.97
Hotel room         1.33
Private room       1.93
Shared room        4.46
Name: accommodates, dtype: float64

In [ ]:
listings.groupby('room_type')[['accommodates','bedrooms','bed']].mean()

KeyError: "Columns not found: 'bed'"

In [ ]:
listings.groupby('price').mean()

TypeError: dtype 'str' does not support operation 'mean'

In [49]:
summary = listings.groupby('room_type').agg(
    n_listings=('id','count'),
    avg_guests=('accommodates','mean'),
    median_reviews=('number_of_reviews','median')
).round(2)
summary

,n_listings,avg_guests,median_reviews
room_type,,,
Entire home/apt,3760,3.97,8.0
Hotel room,3,1.33,56.0
Private room,787,1.93,12.0
Shared room,24,4.46,81.5


Two categories at once: crosstab
how many entire homes, private rooms and other room-types are there in the neighbourhood?

In [50]:
listings.groupby(['neighbourhood_cleansed','room_type']).size().head(8)

neighbourhood_cleansed  room_type      
Bromma                  Entire home/apt    259
                        Private room        22
Enskede-Årsta-Vantörs   Entire home/apt    362
                        Private room        83
Farsta                  Entire home/apt    145
                        Private room        32
Hägersten-Liljeholmens  Entire home/apt    394
                        Private room        70
dtype: int64

In [51]:
pd.crosstab(listings['neighbourhood_cleansed'],listings['room_type'])

room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Bromma,259,0,22,0
Enskede-Årsta-Vantörs,362,0,83,0
Farsta,145,0,32,0
Hägersten-Liljeholmens,394,0,70,0
Hässelby-Vällingby,118,0,30,0
Kungsholmens,332,0,80,8
Norrmalms,373,3,86,7
Rinkeby-Tensta,74,0,33,1
Skarpnäcks,167,0,54,0


Create new columns: np.where

In [52]:
listings['for_groups'] = listings['accommodates'] >= 4
listings[['accommodates','for_groups']].head(5)

,accommodates,for_groups
0,2,False
1,1,False
2,1,False
3,4,True
4,5,True


In [ ]:
import numpy as np
listings['size_type'] = np.where(listings['accommodates'] >= 4,'group','small')
listings['size_type'].value_counts()

size_type
small    2405
group    2169
Name: count, dtype: int64

In [66]:
listings['size_type'] = np.where(listings['size_type'] >= 2, 'small', np.where(listings['size_type'] >= 4, 'medium', 'large'))
listings['size_type'].value_counts()

TypeError: '>=' not supported between instances of 'str' and 'int'

In [54]:
listings.shape

(4574, 85)

copy on write

In [56]:
guests = listings['accommodates']
guest.iloc[0] = 99

print(guests[iloc[0]])
print(listings[0,'accommodates'])

NameError: name 'guest' is not defined

Turn numbers in top groups: cut and qcut

In [67]:
pd.cut(listings['accommodates'],bins=[0,2,4,8,16]).value_counts().sort_index()

accommodates
(0, 2]     1882
(2, 4]     1478
(4, 8]     1117
(8, 16]      97
Name: count, dtype: int64

In [70]:
pd.cut(listings['accommodates'],bins=[0,2,4,8,16]).isna().sum())

SyntaxError: unmatched ')' (145332887.py, line 1)

In [72]:
listings['size_group'] = pd.cut(
    listings['accommodates'],
    bins=[0,2,4,8,16],
    labels=['1-2','3-4','5-8','9-16']
)

print(listings['size_group'].isna().sum())
print(listings['size_group'].value_counts().sort_index())

0
size_group
1-2     1882
3-4     1478
5-8     1117
9-16      97
Name: count, dtype: int64


In [73]:
pd.qcut(listings['number_of_reviews'],4).value_counts().sort_index()

number_of_reviews
(-0.001, 2.0]     1227
(2.0, 9.0]        1083
(9.0, 29.0]       1121
(29.0, 1293.0]    1143
Name: count, dtype: int64

In [74]:
listings['number_of_reviews'].describe()

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64

In [75]:
pd.qcut(listings['availability_365'],4)

ValueError: Bin edges must be unique: Index([0.0, 0.0, 84.0, 277.0, 365.0], dtype='float64', name='availability_365').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [80]:
print((listings['availability_365']==0).sum())

1494


In [81]:
pd.qcut(listings['availability_365'],4,duplicates='drop').value_counts().sort_index()

availability_365
(-0.001, 84.0]    2302
(84.0, 277.0]     1131
(277.0, 365.0]    1141
Name: count, dtype: int64

In [84]:
listings['availability_group'] = pd.qcut(
    listings['availability_365'],
    4,
    labels=['low','medium','high'],
    duplicates='drop'
)
listings['availability_group'].value_counts().sum()

np.int64(4574)

Put all todays learning together

 which neighbourhoods have the highest and lowest share of listings for groups, with room for four or more guests? 
and how many listings is each share based on
1. Group
2.calculate
3.sort
